# 🧹 02 — Data Cleaning & Imputation

**Objective:** Handle missing values, drop high-nullity columns, impute using Season × City medians, and validate the cleaned dataset.

## 2.1 Missing Value Assessment

In [12]:
print(df.isnull().sum())
print(df.shape)
print((df.isnull().sum()/df.shape[0]) *100 )

City              0
Date              0
PM2.5          4598
PM10          11140
NO             3582
NO2            3585
NOx            4185
NH3           10328
CO             2059
SO2            3854
O3             4022
Benzene        5623
Toluene        8041
Xylene        18109
AQI            4681
AQI_Bucket     4681
Year              0
Month             0
Season            0
dtype: int64
(29531, 19)
City           0.000000
Date           0.000000
PM2.5         15.570079
PM10          37.723071
NO            12.129626
NO2           12.139785
NOx           14.171549
NH3           34.973418
CO             6.972334
SO2           13.050692
O3            13.619586
Benzene       19.041008
Toluene       27.229014
Xylene        61.322001
AQI           15.851139
AQI_Bucket    15.851139
Year           0.000000
Month          0.000000
Season         0.000000
dtype: float64


## 2.2 Drop High-Nullity Columns

Dropping `Xylene` (61%), `Toluene` (27%), and `NH3` (35%) due to excessive missing data.

In [13]:
df = df.drop(columns=['Xylene','Toluene','NH3'])
df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,AQI_Bucket,Year,Month,Season
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,0.92,27.64,133.36,0.00,NaN,NaN,2015,1,Winter
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,0.97,24.55,34.06,3.68,NaN,NaN,2015,1,Winter
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,17.40,29.07,30.70,6.80,NaN,NaN,2015,1,Winter
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,1.70,18.59,36.08,4.43,NaN,NaN,2015,1,Winter
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,22.10,39.33,39.31,7.01,NaN,NaN,2015,1,Winter


## 2.3 AQI Bucket Inspection

In [14]:
df_aqi_b=df[df['AQI_Bucket'] != float('nan')]
df_aqi_b.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,AQI_Bucket,Year,Month,Season
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,0.92,27.64,133.36,0.00,NaN,NaN,2015,1,Winter
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,0.97,24.55,34.06,3.68,NaN,NaN,2015,1,Winter
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,17.40,29.07,30.70,6.80,NaN,NaN,2015,1,Winter
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,1.70,18.59,36.08,4.43,NaN,NaN,2015,1,Winter
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,22.10,39.33,39.31,7.01,NaN,NaN,2015,1,Winter


## 2.4 Numerical Imputation — Season × City Median

Strategy: Fill missing pollutant values with the **median** for the same (Season, City) group. If still missing, fall back to the global median.

In [15]:
num_cols = []
for i in df.columns:
    if(df[i].dtype == 'float64' or df[i].dtype == float):
        num_cols.append(i)
for col in num_cols:
  df[col] = df.groupby(['Season','City'])[col].transform(
    lambda x: x.fillna(x.median())
  )
  df[col] = df[col].fillna(df[col].median())

In [16]:
df.isnull().sum()

City             0
Date             0
PM2.5            0
PM10             0
NO               0
NO2              0
NOx              0
CO               0
SO2              0
O3               0
Benzene          0
AQI              0
AQI_Bucket    4681
Year             0
Month            0
Season           0
dtype: int64

## 2.5 AQI Bucket Imputation — Mode

In [17]:
df[df['AQI_Bucket'].notna()].head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,AQI_Bucket,Year,Month,Season
28,Ahmedabad,2015-01-29,83.13,118.44,6.93,28.71,33.72,6.93,49.52,59.76,0.02,209.0,Poor,2015,1,Winter
29,Ahmedabad,2015-01-30,79.84,118.44,13.85,28.68,41.08,13.85,48.49,97.07,0.04,328.0,Very Poor,2015,1,Winter
30,Ahmedabad,2015-01-31,94.52,118.44,24.39,32.66,52.61,24.39,67.39,111.33,0.24,514.0,Severe,2015,1,Winter
31,Ahmedabad,2015-02-01,135.99,118.44,43.48,42.08,84.57,43.48,75.23,102.70,0.40,782.0,Severe,2015,2,Winter
32,Ahmedabad,2015-02-02,178.33,118.44,54.56,35.31,72.80,54.56,55.04,107.38,0.46,914.0,Severe,2015,2,Winter


In [18]:
df['AQI_Bucket'] = df.groupby(['Season','City'])['AQI_Bucket'].transform(
    lambda x : x.fillna(x.mode()[0] if not x.mode().empty else x)
)

## 2.6 Post-Cleaning Validation

In [19]:
df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,AQI_Bucket,Year,Month,Season
0,Ahmedabad,2015-01-01,75.42,118.44,0.92,18.22,17.15,0.92,27.64,133.36,0.00,465.5,Severe,2015,1,Winter
1,Ahmedabad,2015-01-02,75.42,118.44,0.97,15.69,16.46,0.97,24.55,34.06,3.68,465.5,Severe,2015,1,Winter
2,Ahmedabad,2015-01-03,75.42,118.44,17.40,19.30,29.70,17.40,29.07,30.70,6.80,465.5,Severe,2015,1,Winter
3,Ahmedabad,2015-01-04,75.42,118.44,1.70,18.48,17.97,1.70,18.59,36.08,4.43,465.5,Severe,2015,1,Winter
4,Ahmedabad,2015-01-05,75.42,118.44,22.10,21.42,37.76,22.10,39.33,39.31,7.01,465.5,Severe,2015,1,Winter


In [20]:
df['AQI'].value_counts()
df['AQI'].unique()
df['AQI'].nunique()
df.groupby('AQI')['Date'].mean()
df.sort_values('AQI')
df.head()


,City,Date,PM2.5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,AQI_Bucket,Year,Month,Season
0,Ahmedabad,2015-01-01,75.42,118.44,0.92,18.22,17.15,0.92,27.64,133.36,0.00,465.5,Severe,2015,1,Winter
1,Ahmedabad,2015-01-02,75.42,118.44,0.97,15.69,16.46,0.97,24.55,34.06,3.68,465.5,Severe,2015,1,Winter
2,Ahmedabad,2015-01-03,75.42,118.44,17.40,19.30,29.70,17.40,29.07,30.70,6.80,465.5,Severe,2015,1,Winter
3,Ahmedabad,2015-01-04,75.42,118.44,1.70,18.48,17.97,1.70,18.59,36.08,4.43,465.5,Severe,2015,1,Winter
4,Ahmedabad,2015-01-05,75.42,118.44,22.10,21.42,37.76,22.10,39.33,39.31,7.01,465.5,Severe,2015,1,Winter


## 2.7 Duplicate Check

In [21]:
print(df.duplicated().sum())

0
